# Ejercicio 2 - Productos, excepciones y JSON

Vamos a representar productos con clases y evitar que el inventario acepte stock negativo.

**Temas:** clases, objetos, atributos, `@property`, herencia, `super()`, polimorfismo, excepciones y JSON.

## 1. Excepción personalizada

Una excepción es un error que el programa puede lanzar y capturar. Esta clase le da un nombre claro a la regla de negocio del stock.

In [ ]:
class StockInvalidoError(Exception):
    def __init__(self, mensaje="El stock no puede ser un valor negativo."):
        super().__init__(mensaje)

## 2. Clase base

`DispositivoEcologico` representa cualquier producto. `self._stock` usa guion bajo para indicar que se debe acceder al stock mediante la propiedad `stock`. El setter valida cada asignación.

In [ ]:
class DispositivoEcologico:
    def __init__(self, id_producto, nombre, precio, stock):
        self.id_producto = id_producto
        self.nombre = nombre
        self.precio = float(precio)
        self._stock = 0
        self.stock = stock  # pasa por el setter y valida desde el inicio

    @property
    def stock(self):
        return self._stock

    @stock.setter
    def stock(self, valor):
        if valor < 0:
            mensaje = f"Stock inválido para {self.nombre}: se intentó asignar {valor}."
            raise StockInvalidoError(mensaje)
        self._stock = valor

    def generar_ficha_tecnica(self):
        return {
            "id_producto": self.id_producto,
            "nombre": self.nombre,
            "precio": f"${self.precio:.2f}",
            "stock": self.stock
        }

**Cómo leer el setter:** cuando escribimos `producto.stock = 5`, Python ejecuta el método `stock` decorado con `@stock.setter`. Si el valor es negativo, `raise` lanza la excepción; si no, lo guarda en `_stock`.

## 3. Herencia y polimorfismo

`PanelSolar` hereda de `DispositivoEcologico`. `super()` ejecuta la inicialización general. Después, el método `generar_ficha_tecnica` se sobrescribe para sumar los datos propios del panel.

In [ ]:
class PanelSolar(DispositivoEcologico):
    def __init__(self, id_producto, nombre, precio, stock, eficiencia, potencia_w):
        super().__init__(id_producto, nombre, precio, stock)
        self.eficiencia = float(eficiencia)
        self.potencia_w = float(potencia_w)

    def generar_ficha_tecnica(self):
        ficha = super().generar_ficha_tecnica()
        ficha.update({
            "tipo": "Panel Solar Fotovoltaico",
            "potencia_nominal_w": self.potencia_w,
            "eficiencia": f"{self.eficiencia:.1f}%"
        })
        return ficha

In [ ]:
panel = PanelSolar("SOL-001", "Panel EcoTech 450 W", 285000, 12, 21.5, 450)
print(panel.generar_ficha_tecnica())
print("Stock actual:", panel.stock)

La ficha conserva los datos del producto base y suma potencia y eficiencia. Eso es el comportamiento polimórfico pedido.

## 4. Capturar el error de stock

`try` intenta hacer la asignación. `except` recibe el error esperado y muestra una advertencia sin cerrar el programa.

In [ ]:
try:
    panel.stock = -5
except StockInvalidoError as error:
    print("Advertencia controlada:", error)

print("El stock se conserva:", panel.stock)

## 5. Guardar la ficha en JSON

JSON permite guardar datos para volver a leerlos. `with open(...)` abre el archivo y lo cierra automáticamente al terminar.

In [ ]:
class GestorInventario:
    def guardar_catalogo_json(self, lista_dispositivos, ruta_archivo):
        catalogo = []
        for dispositivo in lista_dispositivos:
            ficha = dispositivo.generar_ficha_tecnica()
            catalogo.append(ficha)

        with open(ruta_archivo, "w", encoding="utf-8") as archivo:
            json.dump(catalogo, archivo, ensure_ascii=False, indent=4)

        return ruta_archivo


import json
gestor = GestorInventario()
gestor.guardar_catalogo_json([panel], "catalogo_ecotech.json")

with open("catalogo_ecotech.json", "r", encoding="utf-8") as archivo:
    catalogo_leido = json.load(archivo)

print(catalogo_leido)

En Colab, `catalogo_ecotech.json` se crea en la sesión temporal. En los archivos `.py` de la entrega, la clase `GestorInventario` hace el mismo guardado para una lista de productos.